In [ ]:
# Step-1 Fetch Charged Event from CT

import requests
import json
from openpyxl import Workbook
from openpyxl.utils.exceptions import IllegalCharacterError # Import the specific exception
from google.colab import files
import re # Import the re module

# ---- CONFIG ----
ACCOUNT_ID = "6K5-56W-8K7Z"
PASSCODE = "WHO-JIE-CLEL"
BASE_URL = "https://in1.api.clevertap.com/1/events.json"
OUTPUT_FILE = "Charged from Oct'25.xlsx"

# ---- HARDCODED VALUES ----
EVENT_NAME = "Charged" # Replace with your actual event name
FROM_DATE = "20260201" # Format: YYYYMMDD
TO_DATE = "20260430"
REQUIRED_FIELDS = [] # Set to [] to disable filtering

# Regex to find illegal characters
ILLEGAL_CHARACTERS_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')


# ---- UTILS ----
def flatten_dict(d, parent_key='', sep='.'):
    """
    Recursively flattens a nested dictionary.
    """
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

def clean_string(text):
    """
    Removes illegal characters from a string.
    """
    if isinstance(text, str):
        return ILLEGAL_CHARACTERS_RE.sub('', text)
    return text

def format_identity(identity_obj):
    """
    Helper function to format the identity field by joining all values into a single string.
    """
    if isinstance(identity_obj, dict) and identity_obj is not None:
        return ", ".join(str(v) for v in identity_obj.values())
    return identity_obj or ""

# ---- MAIN FUNCTION ----
def fetch_events(event_name, from_date, to_date):
    """
    Fetches events from the CleverTap API and writes them to an Excel file.
    """
    print(f"[INFO] Fetching events: {event_name}, From: {from_date}, To: {to_date}")
    headers = {
        "X-CleverTap-Account-Id": ACCOUNT_ID,
        "X-CleverTap-Passcode": PASSCODE,
        "Content-Type": "application/json"
    }
    payload = {
        "event_name": event_name,
        "from": int(from_date),
        "to": int(to_date),
        "event_properties": []
    }

    try:
        response = requests.post(f"{BASE_URL}?events=false", headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
    except Exception as e:
        print(f"[ERROR] Failed to fetch initial cursor: {e}")
        return

    print(f"[INFO] Initial API status: {result.get('status')}")
    cursor = result.get("cursor")

    if not cursor:
        print("[ERROR] No cursor returned.")
        print("[DEBUG] Response:", result)
        return

    all_records = []
    fetch_cursor(cursor, event_name, headers, all_records)

    if all_records:
        write_to_excel(all_records, event_name)
    else:
        print("[WARN] No records fetched.")

def fetch_cursor(cursor, event_name, headers, all_records):
    """
    Recursively fetches data using a cursor and appends records.
    """
    while cursor:
        print(f"[INFO] Fetching cursor: {cursor}")
        try:
            response = requests.get(f"{BASE_URL}?cursor={cursor}", headers=headers)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"[ERROR] Failed to fetch data for cursor {cursor}: {e}")
            break

        records = data.get("records", [])
        print(f"[INFO] Records fetched in this page: {len(records)}")

        for record in records:
            # Pre-process the identity field to handle dictionary values
            if 'profile' in record and 'identity' in record['profile']:
                record['profile']['identity'] = format_identity(record['profile']['identity'])

            # Flatten the entire record
            flat_record = flatten_dict(record)
            all_records.append(flat_record)

        cursor = data.get("next_cursor")
        if not cursor:
            print("[INFO] No more pages left.")

def write_to_excel(records, event_name):
    """
    Writes the fetched records to an Excel file.
    """
    print(f"[INFO] Preparing to write {len(records)} rows to {OUTPUT_FILE}")
    wb = Workbook()
    ws = wb.active
    ws.title = event_name[:20]

    # Step 1: Determine headers
    headers = set()
    for record in records:
        headers.update(record.keys())
    headers = sorted(headers)
    ws.append(headers)

    # Step 2: Write filtered rows
    valid_count = 0
    for record in records:
        skip = False
        for field in REQUIRED_FIELDS:
            field_value = record.get(field, "")
            if isinstance(field_value, str):
                field_value = field_value.strip().lower()
            if not field_value or field_value == "na":
                skip = True
                break
        if skip:
            continue

        row = []
        for h in headers:
            value = record.get(h, "")

            # Ensure the value is a string before cleaning and appending.
            if isinstance(value, (list, dict)):
                value = json.dumps(value)
            if value is None:
                value = ""

            # Clean the string value before appending to the row
            value = clean_string(str(value))
            row.append(value)
        ws.append(row)
        valid_count += 1

    wb.save(OUTPUT_FILE)
    print(f"[SUCCESS] File saved: {OUTPUT_FILE} with {valid_count} valid rows")


# ---- RUN THE FETCH ----
fetch_events(EVENT_NAME, FROM_DATE, TO_DATE)


[INFO] Fetching events: Charged, From: 20260201, To: 20260430
[INFO] Initial API status: success
[INFO] Fetching cursor: ZyZjfAEIAQJib2F9Kz8NegUHCARua2N%2Bbmpnek4BBgNuamt6Zm9rewcBTQZmbS57K2oue04ATQQrai57endiNwMHAgxnbGd%2BZGwuLm0BBgNuamt6Zm9rewcBTQVgbWt7bmtjfgsABAUraGN8K2oue04ATQQrai57K2p%2FZgJMAANkYmJ9Ym9hfU5VbgVgbWt7bmtjfgsABAUra2V8bmpregMFCARiay55Zm0ue04ATQQrai57K2ouex8dAUhmbWFzZ2xnfgEGTVEIa2V8bmpregMFCARiay56YG1rewsBAAFuamd6K2hjfE4ATQQrai57K2oue04AHBlnJmN8ZGJifQcFAgIrPw16YG1rewsBAAFuamd6K2tlfAsACAVmb2t7YmsueQMHTQQrai57K2oue04ATQR6d2I3Zm1hcwIGBAFkbC4uCGtlfAsACAVmb2t7YmsuegUHCARua2N%2Bbmpnek4CAAMrai57K2oue04ATQQran9mZyZjfAEIAQJib2F9Kz8NegUHCARua2N%2Bbmpnek4BBgNuamt6Zm9rewcBTQZmbS57K2oue04ATQQrai57endiNwMHAgxnbGd%2BZGwuLm0BBgNuamt6Zm9rewcBTQVgbWt7bmtjfgsABAUraGN8K2oue04ATQQrai57K2p%2FZgJMAANkYmJ9Ym9hfU5VbgVgbWt7bmtjfgsABAUra2V8bmpregMFCARiay55Zm0ue04ATQQrai57K2ouex8dAUhmbWFzZ2xnfgEGTVEIa2V8bmpregMFCARiay56YG1rewsBAAFuamd6K2hjfE4ATQQrai57K2oue04AHBlnJmN8ZGJifQcFAgIrPw16YG1rewsBAAFuamd6K

In [ ]:
!pip install pandas openpyxl tqdm
# =====================================================
# Step-2 Charged Data Cleaning till 5 Items
# =====================================================

import pandas as pd, ast
from tqdm import tqdm
from google.colab import files
from datetime import datetime

print("📁 Upload file")
uploaded = files.upload()
file = list(uploaded.keys())[0]

df = pd.read_excel(file)

clean_rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    try:
        # =========================
        # TS FIX
        # =========================
        ts_val = str(row.get("ts", ""))
        try:
            ts_val = datetime.strptime(ts_val, "%Y%m%d%H%M%S").strftime("%Y-%m-%d")
        except:
            ts_val = ""

        # =========================
        # BASE (MATCH YOUR OUTPUT ORDER)
        # =========================
        base = {
            "identity": row.get("profile.identity", ""),
            "invoice_number": row.get("event_props.Invoice Number", ""),
            "type": "event",
            "event_name": "Charged",
            "api_type": "CleverTapApi",
            "ts": ts_val,

            "Tender Name": row.get("event_props.Tender Name", ""),
            "CUID Number": row.get("event_props.CUID Number", ""),
            "Invoice Number": row.get("event_props.Invoice Number", ""),
            "Invoice Type": row.get("event_props.Invoice Type", ""),
            "Store Code": row.get("event_props.Store Code", ""),
            "Store City": row.get("event_props.Store City", ""),
            "Billing Store Zone": row.get("event_props.Billing Store Zone", ""),
            "Total Billing Value": row.get("event_props.Total Billing Value", ""),
            "Total MRP Value": row.get("event_props.Total MRP Value", "")
        }

        # =========================
        # ITEMS PARSE
        # =========================
        items_raw = row.get("items", "[]")

        try:
            items = ast.literal_eval(items_raw)
        except:
            items = []

        if not isinstance(items, list):
            items = []

        for i, item in enumerate(items[:5], start=1):
            base[f"Item{i}_Billing Discount Percentage"] = item.get("Items|Billing Discount Percentage", "")
            base[f"Item{i}_Billing Value"] = item.get("Items|Billing Value", "")
            base[f"Item{i}_Invoice Brand"] = item.get("Items|Invoice Brand", "")
            base[f"Item{i}_MRP"] = item.get("Items|MRP", "")
            base[f"Item{i}_Model Number"] = item.get("Items|Model Number", "")

        # =========================
        # IDENTITIES (LAST)
        # =========================
        identities = row.get("profile.all_identities", "[]")
        try:
            identities = ast.literal_eval(identities)
        except:
            identities = []

        for i, val in enumerate(identities, start=1):
            base[f"identity_{i}"] = val

        clean_rows.append(base)

    except:
        pass

# =========================
# DF + ORDER
# =========================
clean_df = pd.DataFrame(clean_rows)

# FIX COLUMN ORDER EXACTLY AS YOU WANT
first_cols = [
    "identity","invoice_number","type","event_name","api_type","ts",
    "Tender Name","CUID Number","Invoice Number","Invoice Type",
    "Store Code","Store City","Billing Store Zone",
    "Total Billing Value","Total MRP Value"
]

item_cols = []
for i in range(1,6):
    item_cols += [
        f"Item{i}_Billing Discount Percentage",
        f"Item{i}_Billing Value",
        f"Item{i}_Invoice Brand",
        f"Item{i}_MRP",
        f"Item{i}_Model Number"
    ]

identity_cols = [c for c in clean_df.columns if c.startswith("identity_")]

final_cols = first_cols + item_cols + identity_cols
final_cols = [c for c in final_cols if c in clean_df.columns]

clean_df = clean_df[final_cols]

# =========================
# SAVE
# =========================
output = "Final_Output.xlsx"
clean_df.to_excel(output, index=False)

print(f"✅ Done: {len(clean_df)} rows")
files.download(output)

📁 Upload file


Saving Charged Clean RAW from 2019 to Mar'26.xlsx to Charged Clean RAW from 2019 to Mar'26.xlsx


100%|██████████| 402039/402039 [00:58<00:00, 6875.98it/s]


✅ Done: 402039 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install pandas openpyxl tqdm
# =====================================================
# Step-2.1 Charged Data Cleaning no items limit
# =====================================================

import pandas as pd, ast
from tqdm import tqdm
from google.colab import files
from datetime import datetime

print("📁 Upload file")
uploaded = files.upload()
file = list(uploaded.keys())[0]

df = pd.read_excel(file)

clean_rows = []
max_items = 0  # 🔥 track max items for column creation

for _, row in tqdm(df.iterrows(), total=len(df)):

    try:
        # =========================
        # TS FIX
        # =========================
        ts_val = str(row.get("ts", ""))
        try:
            ts_val = datetime.strptime(ts_val, "%Y%m%d%H%M%S").strftime("%Y-%m-%d")
        except:
            ts_val = ""

        # =========================
        # BASE
        # =========================
        base = {
            "identity": row.get("profile.identity", ""),
            "invoice_number": row.get("event_props.Invoice Number", ""),
            "type": "event",
            "event_name": "Charged",
            "api_type": "CleverTapApi",
            "ts": ts_val,

            "Tender Name": row.get("event_props.Tender Name", ""),
            "CUID Number": row.get("event_props.CUID Number", ""),
            "Invoice Number": row.get("event_props.Invoice Number", ""),
            "Invoice Type": row.get("event_props.Invoice Type", ""),
            "Store Code": row.get("event_props.Store Code", ""),
            "Store City": row.get("event_props.Store City", ""),
            "Billing Store Zone": row.get("event_props.Billing Store Zone", ""),
            "Total Billing Value": row.get("event_props.Total Billing Value", ""),
            "Total MRP Value": row.get("event_props.Total MRP Value", "")
        }

        # =========================
        # ITEMS (NO LIMIT)
        # =========================
        items_raw = row.get("items", "[]")

        try:
            items = ast.literal_eval(items_raw)
        except:
            items = []

        if not isinstance(items, list):
            items = []

        max_items = max(max_items, len(items))  # track max

        for i, item in enumerate(items, start=1):
            base[f"Item{i}_Billing Discount Percentage"] = item.get("Items|Billing Discount Percentage", "")
            base[f"Item{i}_Billing Value"] = item.get("Items|Billing Value", "")
            base[f"Item{i}_Invoice Brand"] = item.get("Items|Invoice Brand", "")
            base[f"Item{i}_MRP"] = item.get("Items|MRP", "")
            base[f"Item{i}_Model Number"] = item.get("Items|Model Number", "")

        # =========================
        # IDENTITIES (LAST)
        # =========================
        identities = row.get("profile.all_identities", "[]")
        try:
            identities = ast.literal_eval(identities)
        except:
            identities = []

        for i, val in enumerate(identities, start=1):
            base[f"identity_{i}"] = val

        clean_rows.append(base)

    except:
        pass

# =========================
# DATAFRAME
# =========================
clean_df = pd.DataFrame(clean_rows)

# =========================
# COLUMN ORDER (DYNAMIC ITEMS)
# =========================
first_cols = [
    "identity","invoice_number","type","event_name","api_type","ts",
    "Tender Name","CUID Number","Invoice Number","Invoice Type",
    "Store Code","Store City","Billing Store Zone",
    "Total Billing Value","Total MRP Value"
]

# 🔥 dynamic item columns
item_cols = []
for i in range(1, max_items + 1):
    item_cols += [
        f"Item{i}_Billing Discount Percentage",
        f"Item{i}_Billing Value",
        f"Item{i}_Invoice Brand",
        f"Item{i}_MRP",
        f"Item{i}_Model Number"
    ]

# identities at last
identity_cols = [c for c in clean_df.columns if c.startswith("identity_")]

final_cols = first_cols + item_cols + identity_cols
final_cols = [c for c in final_cols if c in clean_df.columns]

clean_df = clean_df[final_cols]

# =========================
# SAVE
# =========================
output = "Final_Output_No_Item_Limit.xlsx"
clean_df.to_excel(output, index=False)

print(f"✅ Done: {len(clean_df)} rows | Max Items: {max_items}")
files.download(output)

📁 Upload file


Saving Charged_RAW_Feb_Mar'26.xlsx to Charged_RAW_Feb_Mar'26.xlsx


100%|██████████| 14948/14948 [00:03<00:00, 4704.65it/s]


✅ Done: 14948 rows | Max Items: 46


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install pandas openpyxl
# =====================================================
# Step-5 Implement Sale and Return logic
#identity + brand + model + billing_value
# =====================================================

import pandas as pd
from google.colab import files

# =====================================================
# UPLOAD FILE
# =====================================================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name, dtype=str)

# =====================================================
# ✅ FIX: CLEAN COLUMN NAMES
# =====================================================
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace("event_props.", "", regex=False)

# =====================================================
# NORMALIZE INVOICE TYPE (SALE / RETURN)
# =====================================================
def classify(row):
    t = str(row.get("Invoice Type", "")).lower()

    if "return" in t or "refund" in t:
        return "return"

    try:
        val = row.get("Total Billing Value", "")
        if float(str(val).replace(",", "")) < 0:
            return "return"
    except:
        pass

    return "sale"

df["inv_type_norm"] = df.apply(classify, axis=1)

# =====================================================
# DETECT ITEM COLUMNS (DYNAMIC)
# =====================================================
item_nums = sorted(set([
    int(col.split("_")[0].replace("Item",""))
    for col in df.columns
    if col.startswith("Item") and "Invoice Brand" in col
]))

# =====================================================
# EXPLODE MULTI ITEMS
# =====================================================
items = []

for _, row in df.iterrows():
    for i in item_nums:

        brand = row.get(f"Item{i}_Invoice Brand", "")
        model = row.get(f"Item{i}_Model Number", "")
        billing = row.get(f"Item{i}_Billing Value", "")

        if pd.notna(brand) and str(brand).strip() != "":
            try:
                billing_abs = abs(float(str(billing).replace(",", "")))
            except:
                billing_abs = None

            items.append({
                "orig_index": row.name,
                "identity": row.get("identity", ""),
                "brand": str(brand).strip(),
                "model": str(model).strip(),
                "billing_abs": billing_abs,
                "inv_type": row["inv_type_norm"],
                "ts": row.get("ts", "")
            })

items_df = pd.DataFrame(items)

# =====================================================
# MATCH SALE vs RETURN
# =====================================================
items_df["match_key"] = (
    items_df["identity"].astype(str) + "||" +
    items_df["brand"].astype(str) + "||" +
    items_df["model"].astype(str) + "||" +
    items_df["billing_abs"].astype(str)
)

to_remove = set()

for key, grp in items_df.groupby("match_key"):

    sales = grp[grp["inv_type"] == "sale"].sort_values("ts")
    returns = grp[grp["inv_type"] == "return"].sort_values("ts")

    n = min(len(sales), len(returns))

    if n > 0:
        to_remove.update(sales.iloc[:n]["orig_index"])
        to_remove.update(returns.iloc[:n]["orig_index"])

# =====================================================
# FINAL CLEAN DATA
# =====================================================
clean_df = df.drop(index=list(to_remove)).reset_index(drop=True)
clean_df = clean_df.drop(columns=["inv_type_norm"], errors="ignore")

# =====================================================
# SAVE OUTPUT
# =====================================================
out_name = "cleaned_no_sale_return_multiitem.xlsx"
clean_df.to_excel(out_name, index=False)

files.download(out_name)

print(f"✅ Done | Removed Rows: {len(to_remove)}")

Saving Charged Clean RAW from 2019 to Mar'26.xlsx to Charged Clean RAW from 2019 to Mar'26.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done | Removed Rows: 54995


In [ ]:
# LTV Calculation till 5 items with new to old column

!pip install pandas openpyxl

import pandas as pd
from google.colab import files

# ===============================
# 1. FILE UPLOAD
# ===============================
print("Upload cleaned Excel file:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name)
df.columns = df.columns.str.strip()

print("Columns detected:")
print(df.columns.tolist())


# ===============================
# 2. KEEP SALES ONLY
# ===============================
df = df[df["Invoice Type"].str.lower() == "sale"]


# ===============================
# 3. FIX NUMERIC BILLING
# ===============================
df["Total Billing Value"] = pd.to_numeric(
    df["Total Billing Value"], errors="coerce"
).fillna(0)


# ===============================
# 4. DATE USING ts COLUMN
# ===============================
df["Invoice_Date"] = pd.to_datetime(df["ts"], errors="coerce")


# ===============================
# 5. SORT BY CUSTOMER DATE
# ===============================
df = df.sort_values(["identity", "Invoice_Date"], ascending=[True, False])


# ===============================
# 6. MULTI-ITEM STACKING
# ===============================
def combine_items(row, field):
    vals = []
    for i in range(1, 6):
        col = f"Item{i}_{field}"
        if col in row and pd.notna(row[col]):
            vals.append(str(row[col]))
    return ", ".join(vals)

df["All_Brands"] = df.apply(lambda r: combine_items(r, "Invoice Brand"), axis=1)
df["All_Models"] = df.apply(lambda r: combine_items(r, "Model Number"), axis=1)
df["All_MRP"] = df.apply(lambda r: combine_items(r, "MRP"), axis=1)
df["All_Billing"] = df.apply(lambda r: combine_items(r, "Billing Value"), axis=1)


# ===============================
# 7. ITEM COUNT (NUM PURCHASES)
# ===============================
df["Item_Count"] = df.apply(
    lambda r: sum(pd.notna(r.get(f"Item{i}_Model Number")) for i in range(1, 6)),
    axis=1
)


# ===============================
# 8. MOST FREQUENT STORE LOGIC
# ===============================
def smart_mode(group, col):

    valid = group[col].dropna()

    if valid.empty:
        return ""

    counts = valid.value_counts()

    # repeated store
    if counts.iloc[0] > 1:
        return counts.index[0]

    # all unique → highest billing store
    idx = group["Total Billing Value"].idxmax()
    return group.loc[idx, col]


# ===============================
# 9. CUSTOMER AGGREGATION
# ===============================
def aggregate_customer(group):

    return pd.Series({

        "LTV": group["Total Billing Value"].sum(),
        "Num Purchases": group["Item_Count"].sum(),

        "CUID Number": ", ".join(group["CUID Number"].dropna().astype(str)),
        "CUID Number (Recent)": group["CUID Number"].iloc[0],

        "Last Purchased Store": group["Store Code"].iloc[0],
        "Most Frequent Store": smart_mode(group, "Store Code"),
        "Most Frequent City": smart_mode(group, "Store City"),
        "Most Frequent Zone": smart_mode(group, "Billing Store Zone"),

        "Invoice Brand": ", ".join(group["All_Brands"]),
        "Model Number": ", ".join(group["All_Models"]),
        "MRP": ", ".join(group["All_MRP"]),
        "Billing Value": ", ".join(group["All_Billing"]),
        "Tender Name": ", ".join(group["Tender Name"].dropna().astype(str)),

        "Store Code": ", ".join(group["Store Code"].astype(str)),
        "Store City": ", ".join(group["Store City"].astype(str)),
        "Billing Store Zone": ", ".join(group["Billing Store Zone"].astype(str)),
        "Invoice Numbers": ", ".join(group["Invoice Number"].astype(str)),

        "First Purchase Date": group["Invoice_Date"].min(),
        "Last Purchase Date": group["Invoice_Date"].max()
    })


ltv = df.groupby("identity", dropna=False).apply(aggregate_customer).reset_index()


# ===============================
# 10. DERIVED METRICS
# ===============================
ltv["Avg Purchase Value"] = ltv["LTV"] / ltv["Num Purchases"]

ltv["Customer Lifetime (Days)"] = (
    ltv["Last Purchase Date"] - ltv["First Purchase Date"]
).dt.days

ltv["Months_Since_Last_Purchase"] = (
    pd.Timestamp.today() - ltv["Last Purchase Date"]
).dt.days // 30

ltv["Price Slab"] = pd.cut(
    ltv["LTV"],
    bins=[0, 100000, 300000, 999999999],
    labels=["<1L", "1-3L", ">3L"]
)

ltv.rename(columns={"identity": "Phone"}, inplace=True)
ltv = ltv.sort_values("LTV", ascending=False)


# ===============================
# 11. EXPORT FILE
# ===============================
output_file = "Final_LTV_Output.xlsx"
ltv.to_excel(output_file, index=False)

files.download(output_file)

print("✅ LTV Calculation Completed Successfully")

Upload cleaned Excel file:


Saving cleaned_no_sale_return_multiitem_2019_mar'26.xlsx to cleaned_no_sale_return_multiitem_2019_mar'26.xlsx
Columns detected:
['identity', 'invoice_number', 'type', 'event_name', 'api_type', 'ts', 'Tender Name', 'CUID Number', 'Invoice Number', 'Invoice Type', 'Store Code', 'Store City', 'Billing Store Zone', 'Total Billing Value', 'Total MRP Value', 'Item1_Billing Discount Percentage', 'Item1_Billing Value', 'Item1_Invoice Brand', 'Item1_MRP', 'Item1_Model Number', 'Item2_Billing Discount Percentage', 'Item2_Billing Value', 'Item2_Invoice Brand', 'Item2_MRP', 'Item2_Model Number', 'Item3_Billing Discount Percentage', 'Item3_Billing Value', 'Item3_Invoice Brand', 'Item3_MRP', 'Item3_Model Number', 'Item4_Billing Discount Percentage', 'Item4_Billing Value', 'Item4_Invoice Brand', 'Item4_MRP', 'Item4_Model Number', 'Item5_Billing Discount Percentage', 'Item5_Billing Value', 'Item5_Invoice Brand', 'Item5_MRP', 'Item5_Model Number', 'identity_1', 'identity_2', 'identity_3']


/tmp/ipykernel_1429/2217513808.py:129: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ltv = df.groupby("identity", dropna=False).apply(aggregate_customer).reset_index()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ LTV Calculation Completed Successfully


In [ ]:
#Profile All Identity Cleaning
!pip install openpyxl


import pandas as pd
import re
from google.colab import files

# 1️⃣ Upload file
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 2️⃣ Read file
if file_name.endswith(".csv"):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name)

# 🔹 Column name exactly as in your sheet
raw_column_name = "profile.all_identities"

def clean_phone(phone):
    # Remove everything except digits
    phone = re.sub(r'[^0-9]', '', phone)

    # Remove leading 91 if present and length > 10
    if phone.startswith("91") and len(phone) > 10:
        phone = phone[-10:]

    # Keep only valid 10-digit numbers
    if len(phone) == 10:
        return phone
    else:
        return None

def extract_details(text):
    if pd.isna(text):
        return pd.Series(["", "", "", ""])

    cleaned = re.sub(r'[\[\]"]', '', str(text))
    cleaned = cleaned.replace(".0", "")

    items = [item.strip() for item in cleaned.split(",")]

    emails = []
    phones = []

    for item in items:
        if "@" in item:
            emails.append(item)
        else:
            cleaned_phone = clean_phone(item)
            if cleaned_phone:
                phones.append(cleaned_phone)

    # Max 3 phones
    phone1 = phones[0] if len(phones) > 0 else ""
    phone2 = phones[1] if len(phones) > 1 else ""
    phone3 = phones[2] if len(phones) > 2 else ""

    return pd.Series([
        ", ".join(emails),
        phone1,
        phone2,
        phone3
    ])

# 3️⃣ Apply extraction (no row removed)
df[["Extracted_Emails", "Phone1", "Phone2", "Phone3"]] = df[raw_column_name].apply(extract_details)

# 4️⃣ Save output
output_file = "cleaned_output.xlsx"
df.to_excel(output_file, index=False)

# 5️⃣ Download
files.download(output_file)

Saving Profile.xlsx to Profile.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =========================================
# STEP 1: Upload File
# =========================================
from google.colab import files
uploaded = files.upload()

# =========================================
# STEP 2: Load & Clean Data
# =========================================
import pandas as pd
import numpy as np

file_name = list(uploaded.keys())[0]

if file_name.endswith('.csv'):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name, engine='openpyxl')

# -------- CLEANING --------
df['identity'] = (
    df['identity']
    .astype(str)
    .str.replace('.0','', regex=False)
    .str.strip()
)

df['ts'] = pd.to_datetime(df['ts'], errors='coerce')

df['Financial Year Flag'] = df['Financial Year Flag'].astype(str).str.strip()
df['Last 12 Month Purchased'] = df['Last 12 Month Purchased'].astype(str).str.strip()

df['Total Billing Value'] = pd.to_numeric(df['Total Billing Value'], errors='coerce').fillna(0)

# =========================================
# STEP 3: EXCLUDE IDENTITIES
# =========================================
exclude_ids = [
    "1111119999999",
    "1119999999",
    "9999988800",
    "353777693"
]

df = df[~df['identity'].isin(exclude_ids)]

print("After exclusion - rows:", len(df))
print("Unique customers:", df['identity'].nunique())

# =========================================
# STEP 4: CUSTOMER MASTER
# =========================================
customer = df.groupby('identity', as_index=False).agg(
    Total_LTV=('Total Billing Value', 'sum'),
    Total_Purchases=('identity', 'count'),
    First_Purchase=('ts', 'min'),
    Last_Purchase=('ts', 'max')
)

# =========================================
# STEP 5: LAST 12M METRICS
# =========================================
df_12m = df[df['Last 12 Month Purchased'] == 'Last 12 Month']

cust_12m = df_12m.groupby('identity', as_index=False).agg(
    Last_12M_LTV=('Total Billing Value', 'sum'),
    Last_12M_Purchases=('identity', 'count')
)

customer = customer.merge(cust_12m, on='identity', how='left')

# =========================================
# STEP 6: LAST FY METRICS (FIXED)
# =========================================
FY_TARGET = "FY 2025-26"

df_fy = df[df['Financial Year Flag'] == FY_TARGET]

cust_fy = df_fy.groupby('identity', as_index=False).agg(
    Last_FY_Billing=('Total Billing Value', 'sum'),
    Last_FY_Purchases=('identity', 'count')
)

customer = customer.merge(cust_fy, on='identity', how='left')

# =========================================
# STEP 7: FILL NULLS
# =========================================
customer[['Last_12M_LTV','Last_12M_Purchases',
          'Last_FY_Billing','Last_FY_Purchases']] = \
customer[['Last_12M_LTV','Last_12M_Purchases',
          'Last_FY_Billing','Last_FY_Purchases']].fillna(0)

# =========================================
# STEP 8: DERIVED METRICS
# =========================================
customer['Avg_LTV'] = np.where(
    customer['Total_Purchases'] > 0,
    customer['Total_LTV'] / customer['Total_Purchases'],
    0
)

customer['Avg_Last_FY'] = np.where(
    customer['Last_FY_Purchases'] > 0,
    customer['Last_FY_Billing'] / customer['Last_FY_Purchases'],
    0
)

customer['Active_Flag'] = np.where(
    customer['Last_12M_LTV'] > 0,
    'Active',
    'Inactive'
)

# =========================================
# STEP 9: LTV BUCKETS
# =========================================
quantiles = {
    "Top 50%": 0.5,
    "Top 25%": 0.75,
    "Top 15%": 0.85,
    "Top 10%": 0.9,
    "Top 5%": 0.95,
    "Top 2.5%": 0.975,
    "Top 1%": 0.99
}

cutoffs = {k: customer['Total_LTV'].quantile(v) for k,v in quantiles.items()}

def assign_bucket(x):
    if x >= cutoffs["Top 1%"]:
        return "Top 1%"
    elif x >= cutoffs["Top 2.5%"]:
        return "Top 2.5%"
    elif x >= cutoffs["Top 5%"]:
        return "Top 5%"
    elif x >= cutoffs["Top 10%"]:
        return "Top 10%"
    elif x >= cutoffs["Top 15%"]:
        return "Top 15%"
    elif x >= cutoffs["Top 25%"]:
        return "Top 25%"
    elif x >= cutoffs["Top 50%"]:
        return "Top 50%"
    else:
        return "Bottom 50%"

customer['Bucket'] = customer['Total_LTV'].apply(assign_bucket)

# =========================================
# STEP 10: DASHBOARD SUMMARY
# =========================================
summary_rows = []

bucket_order = [
    "Total","Top 50%","Top 25%","Top 15%",
    "Top 10%","Top 5%","Top 2.5%","Top 1%"
]

for b in bucket_order:
    if b == "Total":
        subset = customer.copy()
    else:
        subset = customer[customer['Bucket'] == b]

    total_ltv = subset['Total_LTV'].sum()
    cx_count = len(subset)
    total_purch = subset['Total_Purchases'].sum()
    avg_ltv = total_ltv / total_purch if total_purch > 0 else 0

    last_fy_bill = subset['Last_FY_Billing'].sum()
    cx_last_year = (subset['Last_FY_Billing'] > 0).sum()
    purch_last_year = subset['Last_FY_Purchases'].sum()
    avg_last_year = last_fy_bill / purch_last_year if purch_last_year > 0 else 0

    summary_rows.append([
        b,
        total_ltv,
        cx_count,
        total_purch,
        avg_ltv,
        last_fy_bill,
        cx_last_year,
        purch_last_year,
        avg_last_year
    ])

summary = pd.DataFrame(summary_rows, columns=[
    "Brackets",
    "Total LTV Billing Value",
    "Count of Cx",
    "No of Purchased",
    "Average Sales Value LTV",
    "Last Year Billing Value",
    "Cx count Last Year",
    "No. of Purchased Last Year",
    "Average Sales Value Last Year"
])

# =========================================
# STEP 11: LTV %
# =========================================
total_ltv_all = summary.loc[summary['Brackets']=="Total","Total LTV Billing Value"].values[0]

summary['LTV Billing %'] = summary['Total LTV Billing Value'] / total_ltv_all

# =========================================
# STEP 12: EXPORT
# =========================================
with pd.ExcelWriter("Final_Dashboard.xlsx") as writer:
    customer.to_excel(writer, sheet_name="Customer_Master", index=False)
    summary.to_excel(writer, sheet_name="Dashboard", index=False)

print("✅ Final Dashboard Created Successfully!")

Saving Last Year Breakup Data.xlsx to Last Year Breakup Data.xlsx
After exclusion - rows: 305573
Unique customers: 210406
✅ Final Dashboard Created Successfully!


In [ ]:
# ================================
# Transforms raw invoice data into clean,
#item-level dataset with extracted SKU,
#deduplicated phone/email, and formatted datetime
# ================================
import pandas as pd
import json
import re
from google.colab import files

# ================================
# UPLOAD FILE
# ================================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

if file_name.endswith('.csv'):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name)

print("File Loaded:", df.shape)

# ================================
# HELPER FUNCTIONS
# ================================
def clean_phone(val):
    if pd.isna(val):
        return None
    val = str(val)
    val = re.sub(r'\.0$', '', val)
    val = re.sub(r'\D', '', val)
    return val[-10:] if len(val) >= 10 else None

def extract_email(val):
    if pd.isna(val):
        return None
    val = str(val)
    return val.lower() if "@" in val else None

def safe_json(x):
    try:
        return json.loads(x)
    except:
        return []

# ================================
# EXPLODE ITEMS
# ================================
df['items_parsed'] = df['items'].apply(safe_json)
df = df.explode('items_parsed').reset_index(drop=True)

# ================================
# ITEM EXTRACTION
# ================================
df['Invoice Brand'] = df['items_parsed'].apply(lambda x: x.get('Items|Invoice Brand') if isinstance(x, dict) else None)
df['Model Number'] = df['items_parsed'].apply(lambda x: x.get('Items|Model Number') if isinstance(x, dict) else None)
df['MRP'] = df['items_parsed'].apply(lambda x: x.get('Items|MRP') if isinstance(x, dict) else None)
df['Billing Value'] = df['items_parsed'].apply(lambda x: x.get('Items|Billing Value') if isinstance(x, dict) else None)
df['Billing Discount Percentage'] = df['items_parsed'].apply(lambda x: x.get('Items|Billing Discount Percentage') if isinstance(x, dict) else None)

# ================================
# EXTRACT ALL CONTACTS
# ================================
def extract_all_contacts(row):
    phones = []
    emails = []

    # profile.all_identities
    try:
        ids = json.loads(row['profile.all_identities'])
    except:
        ids = []

    for i in ids:
        if "@" in str(i):
            emails.append(str(i).lower())
        else:
            phone = clean_phone(i)
            if phone:
                phones.append(phone)

    # profile.phone
    if pd.notna(row.get('profile.phone')):
        phone = clean_phone(row['profile.phone'])
        if phone:
            phones.append(phone)

    # profile.identity
    if pd.notna(row.get('profile.identity')):
        phone = clean_phone(row['profile.identity'])
        if phone:
            phones.append(phone)

    # profile.email
    if pd.notna(row.get('profile.email')):
        email = extract_email(row['profile.email'])
        if email:
            emails.append(email)

    return list(set(phones)), list(set(emails))

df[['phones', 'emails']] = df.apply(lambda row: pd.Series(extract_all_contacts(row)), axis=1)

# ================================
# NAME EXTRACTION
# ================================
def extract_names(row):
    names = []

    if pd.notna(row.get('profile.name')):
        names.append(str(row['profile.name']).strip())

    if pd.notna(row.get('profile.profileData.customername')):
        names.append(str(row['profile.profileData.customername']).strip())

    # remove duplicates (case insensitive)
    names = list(set([n.lower() for n in names if n]))

    return names

df['names'] = df.apply(extract_names, axis=1)

# ================================
# DYNAMIC COLUMNS
# ================================
max_phones = df['phones'].apply(len).max()
max_emails = df['emails'].apply(len).max()
max_names = df['names'].apply(len).max()

for i in range(max_phones):
    df[f'phone{i+1}'] = df['phones'].apply(lambda x: x[i] if i < len(x) else None)

for i in range(max_emails):
    df[f'email{i+1}'] = df['emails'].apply(lambda x: x[i] if i < len(x) else None)

for i in range(max_names):
    df[f'name{i+1}'] = df['names'].apply(lambda x: x[i] if i < len(x) else None)

# ================================
# PRIMARY NAME
# ================================
df['primary_name'] = df['name1']

# ================================
# DATE CONVERSION
# ================================
df['invoice_datetime'] = pd.to_datetime(df['event_props.Invoice Date'], unit='s', errors='coerce')
df['final_datetime'] = pd.to_datetime(df['ts'], format='%Y%m%d%H%M%S', errors='coerce')

# ================================
# FINAL OUTPUT
# ================================
base_cols = [
    'event_props.CUID Number',
    'event_props.Billing Store Zone',
    'event_props.Invoice Number',
    'event_props.Invoice Type',
    'event_props.Store City',
    'event_props.Store Code',
    'event_props.Tender Name',
    'Invoice Brand',
    'Model Number',
    'MRP',
    'Billing Value',
    'Billing Discount Percentage',
    'primary_name',
    'invoice_datetime',
    'final_datetime'
]

phone_cols = [col for col in df.columns if col.startswith('phone')]
email_cols = [col for col in df.columns if col.startswith('email')]
name_cols = [col for col in df.columns if col.startswith('name')]

final_df = df[base_cols + name_cols + phone_cols + email_cols]

# ================================
# EXPORT
# ================================
final_df.to_csv("final_cleaned_data.csv", index=False)
files.download("final_cleaned_data.csv")

print("✅ Done: Full cleaned dataset with dynamic contacts + primary name ready")

Saving Charged from 2019 to Apr'26.xlsx to Charged from 2019 to Apr'26 (1).xlsx
File Loaded: (414685, 36)


/tmp/ipykernel_5505/3922365086.py:146: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['invoice_datetime'] = pd.to_datetime(df['event_props.Invoice Date'], unit='s', errors='coerce')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done: Full cleaned dataset with dynamic contacts + primary name ready


In [ ]:
# ================================
# Match sale & return at item level
# ================================
import pandas as pd
import json
import re
from google.colab import files

# ================================
# UPLOAD FILE
# ================================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

if file_name.endswith('.csv'):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name)

print("File Loaded:", df.shape)

# ================================
# HELPER FUNCTIONS
# ================================
def clean_phone(val):
    if pd.isna(val):
        return None
    val = str(val)
    val = re.sub(r'\.0$', '', val)
    val = re.sub(r'\D', '', val)
    return val[-10:] if len(val) >= 10 else None

def extract_email(val):
    if pd.isna(val):
        return None
    val = str(val)
    return val.lower() if "@" in val else None

def safe_json(x):
    try:
        return json.loads(x)
    except:
        return []

# ================================
# CLASSIFY SALE / RETURN
# ================================
def classify(row):
    t = str(row.get("event_props.Invoice Type", "")).lower()

    if "return" in t or "refund" in t:
        return "return"

    try:
        val = row.get("event_props.Total Billing Value", "")
        if float(str(val).replace(",", "")) < 0:
            return "return"
    except:
        pass

    return "sale"

df["inv_type_norm"] = df.apply(classify, axis=1)

# ================================
# EXPLODE ITEMS
# ================================
df['items_parsed'] = df['items'].apply(safe_json)
df = df.explode('items_parsed').reset_index(drop=True)

# ================================
# ITEM EXTRACTION
# ================================
df['Invoice Brand'] = df['items_parsed'].apply(lambda x: x.get('Items|Invoice Brand') if isinstance(x, dict) else None)
df['Model Number'] = df['items_parsed'].apply(lambda x: x.get('Items|Model Number') if isinstance(x, dict) else None)
df['MRP'] = df['items_parsed'].apply(lambda x: x.get('Items|MRP') if isinstance(x, dict) else None)
df['Billing Value'] = df['items_parsed'].apply(lambda x: x.get('Items|Billing Value') if isinstance(x, dict) else None)
df['Billing Discount Percentage'] = df['items_parsed'].apply(lambda x: x.get('Items|Billing Discount Percentage') if isinstance(x, dict) else None)

# ================================
# CREATE MATCH KEY (IMPORTANT)
# ================================
df['match_key'] = (
    df['event_props.CUID Number'].astype(str) + "||" +
    df['Invoice Brand'].astype(str) + "||" +
    df['Model Number'].astype(str)
)

# ================================
# SALE vs RETURN MATCHING
# ================================
df['Billing Value'] = pd.to_numeric(df['Billing Value'], errors='coerce').abs()

to_remove = []

for key, grp in df.groupby('match_key'):

    sales = grp[grp['inv_type_norm'] == 'sale'].sort_values('ts')
    returns = grp[grp['inv_type_norm'] == 'return'].sort_values('ts')

    n = min(len(sales), len(returns))

    if n > 0:
        to_remove.extend(sales.iloc[:n].index)
        to_remove.extend(returns.iloc[:n].index)

# REMOVE MATCHED SALE + RETURN
df = df.drop(index=to_remove).reset_index(drop=True)

print("Removed Sale-Return matched rows:", len(to_remove))

# ================================
# CONTACT EXTRACTION
# ================================
def extract_all_contacts(row):
    phones, emails = [], []

    try:
        ids = json.loads(row['profile.all_identities'])
    except:
        ids = []

    for i in ids:
        if "@" in str(i):
            emails.append(str(i).lower())
        else:
            phone = clean_phone(i)
            if phone:
                phones.append(phone)

    if pd.notna(row.get('profile.phone')):
        p = clean_phone(row['profile.phone'])
        if p:
            phones.append(p)

    if pd.notna(row.get('profile.identity')):
        p = clean_phone(row['profile.identity'])
        if p:
            phones.append(p)

    if pd.notna(row.get('profile.email')):
        e = extract_email(row['profile.email'])
        if e:
            emails.append(e)

    return list(set(phones)), list(set(emails))

df[['phones', 'emails']] = df.apply(lambda r: pd.Series(extract_all_contacts(r)), axis=1)

# ================================
# NAME EXTRACTION
# ================================
def extract_names(row):
    names = []

    if pd.notna(row.get('profile.name')):
        names.append(str(row['profile.name']).strip())

    if pd.notna(row.get('profile.profileData.customername')):
        names.append(str(row['profile.profileData.customername']).strip())

    return list(set([n.lower() for n in names if n]))

df['names'] = df.apply(extract_names, axis=1)

# ================================
# DYNAMIC COLUMNS
# ================================
max_phones = df['phones'].apply(len).max()
max_emails = df['emails'].apply(len).max()
max_names = df['names'].apply(len).max()

for i in range(max_phones):
    df[f'phone{i+1}'] = df['phones'].apply(lambda x: x[i] if i < len(x) else None)

for i in range(max_emails):
    df[f'email{i+1}'] = df['emails'].apply(lambda x: x[i] if i < len(x) else None)

for i in range(max_names):
    df[f'name{i+1}'] = df['names'].apply(lambda x: x[i] if i < len(x) else None)

df['primary_name'] = df['name1']

# ================================
# DATE CONVERSION
# ================================
df['invoice_datetime'] = pd.to_datetime(df['event_props.Invoice Date'], unit='s', errors='coerce')
df['final_datetime'] = pd.to_datetime(df['ts'], format='%Y%m%d%H%M%S', errors='coerce')

# ================================
# FINAL OUTPUT
# ================================
base_cols = [
    'event_props.CUID Number',
    'event_props.Billing Store Zone',
    'event_props.Invoice Number',
    'event_props.Invoice Type',
    'event_props.Store City',
    'event_props.Store Code',
    'event_props.Tender Name',
    'Invoice Brand',
    'Model Number',
    'MRP',
    'Billing Value',
    'Billing Discount Percentage',
    'primary_name',
    'invoice_datetime',
    'final_datetime'
]

phone_cols = [c for c in df.columns if c.startswith('phone')]
email_cols = [c for c in df.columns if c.startswith('email')]
name_cols = [c for c in df.columns if c.startswith('name')]

final_df = df[base_cols + name_cols + phone_cols + email_cols]

# ================================
# EXPORT
# ================================
final_df.to_csv("final_cleaned_no_returns.csv", index=False)
files.download("final_cleaned_no_returns.csv")

print("✅ Done: Sale-return adjusted dataset ready")

Saving Charged RAW from 2019 to Apr'26.xlsx to Charged RAW from 2019 to Apr'26.xlsx
File Loaded: (414685, 36)
Removed Sale-Return matched rows: 67358


/tmp/ipykernel_41565/3792406369.py:187: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['invoice_datetime'] = pd.to_datetime(df['event_props.Invoice Date'], unit='s', errors='coerce')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done: Sale-return adjusted dataset ready
